# Scene Index Quickstart with VideoDB

This guide introduces Scene Indexing with the new VideoDB understanding and indexing workflow.

Scene indexing lets you describe what is visible in a video, make those descriptions searchable, and retrieve playable moments from visual queries.

In this notebook, you will:

- Upload a sample video.
- Run `understand(vlm)` to create visual scene descriptions.
- Configure scene segmentation, frame sampling, and the VLM prompt.
- Index the VLM artifact directly with `video.index()`.
- Search the scene index with `semantic_search()`.
- Generate a playable stream from matching timestamps.
- Inspect the indexes created on the video.

## 1. Install Dependencies

In [ ]:
!pip install -q videodb python-dotenv


## 2. Connect to VideoDB

Enter your VideoDB API key when prompted.

Enter your VideoDB API key when prompted. You can get one from the [VideoDB Console](https://console.videodb.io). Get $20 free credits. **No credit card needed**.


In [3]:
import os
import time
from getpass import getpass
from uuid import uuid4

from IPython.display import display
from videodb import connect, play_stream

os.environ["VIDEO_DB_API_KEY"] = getpass("Please enter your VideoDB API Key: ")

conn = connect()
coll = conn.get_collection()

print("Connected to VideoDB successfully.")

Please enter your VideoDB API Key: ··········
Connected to VideoDB successfully.


## 3. Upload a Video

This quickstart uses a short educational video about the solar system. A compact sample keeps the visual indexing flow fast while still containing rich scenes: stars, the Milky Way, planets, diagrams, labels, and space footage.

In [4]:
video = coll.upload(url="https://www.youtube.com/watch?v=libKVRa01L8")
print("Video ID:", video.id)

preview_stream = video.generate_stream()
display(play_stream(preview_stream))

Video ID: m-z-019f3ab1-4f0f-7142-b9a8-e9a43b188695


## 4. Understand Visual Scenes

The `vlm` analyzer uses a vision-language model to describe the visual content in each video segment.

The key scene-indexing controls are:

- `segmentation`: how the video is split into visual records. Here we use 10-second time windows.
- `sampling`: which frames are sampled from each segment. Here we sample two frames uniformly.
- `prompt`: what the VLM should describe for each segment.
- `schema`: the structured fields we want back from the analyzer.

In [5]:
visual_prompt = (
    "Describe the visible scene in 1-2 concise sentences for visual search. "
    "Mention important space visuals, planets, galaxies, diagrams, labels, objects, and actions."
)

visual_understanding = video.understand(
    segmentation={
        "type": "time",
        "seconds": 10,
    },
    analyzers=[
        {
            "type": "vlm",
            "name": "visual_scenes",
            "sampling": {
                "strategy": "uniform",
                "frame_count": 2,
            },
            "config": {
                "model": "pro",
                "prompt": visual_prompt,
                "schema": {
                    "scene_description": "string",
                    "primary_visual_topic": "string",
                },
            },
        }
    ],
)

visual_understanding.wait_until_complete(timeout=3600, poll_interval=15)
visual_output = visual_understanding.get_analyzer("visual_scenes").get_output()

print("Visual understanding status:", visual_output.get("status"))
print("Visual scenes:", len(visual_output.get("scenes", [])))

Visual understanding status: done
Visual scenes: 26


## 5. Preview Scene Descriptions

Inspect a few analyzer records before indexing. This is useful when tuning the segmentation, sampling, prompt, or schema for your own use case.

In [6]:
for scene in visual_output.get("scenes", [])[:5]:
    data = scene.get("data", {}) or {}
    print(f"{scene.get('start'):.2f}s - {scene.get('end'):.2f}s")
    print("Description:", data.get("scene_description") or data.get("description") or data.get("text") or "")
    print("Topic:", data.get("primary_visual_topic", ""))
    print("----")

0.00s - 10.00s
Description: A dense starfield with the Milky Way band fills the background while large white text labels read 'OUR SOLAR SYSTEM' and 'ONE OF OVER 500 IN THE MILKY WAY GALAXY' across the sky. A black lower banner displays the program title 'SCIENCE 101 | THE SOLAR SYSTEM' with a small yellow National Geographic logo at the corner.
Topic: Title card over Milky Way starfield labeling the solar system and the Science 101 series
----
10.00s - 20.00s
Description: A dense Milky Way starfield fills the sky above Earth's curved limb with a pinkish auroral glow, while parts of the ISS solar panels and structure appear at the top left. On-screen caption labels it as a view from the International Space Station.
Topic: Milky Way and Earth's horizon as seen from the International Space Station (ISS)
----
20.00s - 30.00s
Description: A spacecraft-orbit view shows Earth’s curved limb and solar panels against a star-filled sky with the Milky Way stretching overhead; the next shot is a l

## 6. Create the Scene Index

Create an index directly from the VLM artifact. This keeps the quickstart aligned with the core workflow: understand the video, index the artifact fields, then retrieve matching moments.

- `use_for=["semantic", "query"]` makes the scene artifact available for semantic search and structured filtering.
- `fields["semantic"]` controls which fields are embedded for semantic retrieval.
- `fields["text"]` controls which fields can be returned as result metadata.
- `fields["filter"]` controls which fields can be used with exact filters in `query()`.

In [7]:
VISUAL_INDEX_NAME = f"scene_quickstart_{uuid4().hex[:8]}"
INDEX_READY_STATUSES = {"ready", "done"}
INDEX_ACTIVE_STATUSES = {"building", "processing"}


def wait_until_index_ready(video, index, timeout=1800, poll_interval=10):
    deadline = time.time() + timeout
    latest_index = index

    while time.time() < deadline:
        status = getattr(latest_index, "status", None)

        if status in INDEX_READY_STATUSES:
            return latest_index

        if status and status not in INDEX_ACTIVE_STATUSES:
            raise RuntimeError(f"Index build ended with status: {status}")

        time.sleep(poll_interval)

        try:
            latest_index = video.get_index(index_id=index.index_id)
        except Exception as exc:
            if "not found" in str(exc).lower():
                continue
            raise

    raise TimeoutError(f"Index was not ready within {timeout} seconds.")


visual_index = video.index(
    name=VISUAL_INDEX_NAME,
    source=visual_output,
    use_for=["semantic", "query"],
    fields={
        "semantic": ["scene_description", "primary_visual_topic"],
        "text": ["scene_description", "primary_visual_topic"],
        "filter": ["primary_visual_topic"],
    },
)

visual_index = wait_until_index_ready(video, visual_index)

print("Visual index name:", VISUAL_INDEX_NAME)
print("Visual index ID:", visual_index.index_id)
print("Visual index status:", visual_index.status)
print("Index fields:", visual_index.fields)

Visual index name: scene_quickstart_221d0ecd
Visual index ID: 841ed791e6374609
Visual index status: ready
Index fields: {'filter': ['primary_visual_topic'], 'semantic': ['scene_description', 'primary_visual_topic'], 'text': ['scene_description', 'primary_visual_topic']}


## 7. Search Visual Moments

Use `semantic_search()` when you want to find moments by visual meaning. The query does not need to match the exact words in the scene descriptions.

The search result gives us the matching timestamps. We use those timestamps to show the corresponding scene details from the VLM artifact previewed earlier.

In [8]:
def get_scene_data_for_shot(shot, visual_output):
    for scene in visual_output.get("scenes", []):
        same_start = abs(float(scene.get("start", -1)) - float(shot.start)) < 0.01
        same_end = abs(float(scene.get("end", -1)) - float(shot.end)) < 0.01

        if same_start and same_end:
            return scene.get("data", {}) or {}

    return {}


visual_results = video.semantic_search(
    query="Milky Way galaxy, solar system diagram, stars, planets, and space visuals",
    index_names=[VISUAL_INDEX_NAME],
    top_k=5,
    return_fields=["scene_description", "primary_visual_topic"],
)

for shot in visual_results.shots:
    scene_data = get_scene_data_for_shot(shot, visual_output)
    print(f"{shot.start:.2f}s - {shot.end:.2f}s")
    print("Topic:", scene_data.get("primary_visual_topic", ""))
    print("Description:", scene_data.get("scene_description", ""))
    print("----")

20.00s - 30.00s
Topic: Milky Way galaxy diagram and orbital view showing the Solar System location
Description: A spacecraft-orbit view shows Earth’s curved limb and solar panels against a star-filled sky with the Milky Way stretching overhead; the next shot is a labeled spiral diagram of the Milky Way with the Solar System location indicated. The diagram prominently displays text labels ‘MILKY WAY GALAXY’ and ‘SOLAR SYSTEM.’
----
30.00s - 40.00s
Topic: Milky Way galaxy illustration with overlaid statistic and labeled Sun inset
Description: A stylized spiral Milky Way galaxy with a bright central bulge and swirling arms is overlaid by large white text reading "ONLY 15% OF STARS HOST PLANETS." An inset red close-up of the Sun labeled "OUR SUN" is shown with a callout line pointing to a location in the galaxy (National Geographic logo and NASA/JPL credit visible).
----
0.00s - 10.00s
Topic: Title card over Milky Way starfield labeling the solar system and the Science 101 series
Descripti

## 8. Play Search Results

Generate a stream from the matching timestamps and explicitly display the player.

In [9]:
result_timeline = [(shot.start, shot.end) for shot in visual_results.shots]

if result_timeline:
    print("Result timeline:", result_timeline)
    stream_link = video.generate_stream(result_timeline)
    player = play_stream(stream_link)
    display(player)
else:
    print("No visual moments found. Try a broader visual query.")

Result timeline: [(20.0, 30.0), (30.0, 40.0), (0.0, 10.0), (180.0, 190.0), (220.0, 230.0)]


## 9. Try Another Query

Scene indexes are useful because you can ask different visual questions without re-running the visual understanding step.

In [10]:
planet_results = video.semantic_search(
    query="terrestrial planets Mercury Venus Earth Mars shown in an educational graphic",
    index_names=[VISUAL_INDEX_NAME],
    top_k=3,
    return_fields=["scene_description", "primary_visual_topic"],
)

for shot in planet_results.shots:
    scene_data = get_scene_data_for_shot(shot, visual_output)
    print(f"{shot.start:.2f}s - {shot.end:.2f}s")
    print("Topic:", scene_data.get("primary_visual_topic", ""))
    print("Description:", scene_data.get("scene_description", "")[:300])
    print("----")

50.00s - 60.00s
Topic: Illustration of terrestrial planets (Mercury, Venus, Earth, Mars) with informational overlay
Description: Four labeled planets — Mercury, Venus, Earth, and Mars — are displayed across a starfield background with a black informational panel reading “TERRESTRIAL.” The overlay shows bullet points “Made of rocky material” and “Surfaces are solid,” plus a small “Not to scale” note and a National Geographic l
----
60.00s - 70.00s
Topic: Terrestrial planets diagram and Mercury transiting the Sun
Description: A labeled diagram of four terrestrial planets (Mercury, Venus, Earth, Mars) appears over a starfield with a black info panel listing bullet-point traits (made of rocky material, surfaces solid, don’t have rings, very few moons, relatively small). Below that, a close-up image of the Sun titled 'MERCU
----
40.00s - 50.00s
Topic: Labeled solar system planet lineup with overlaid title card "2 CATEGORIES".
Description: A horizontal lineup of the eight labeled planets (Me

## 10. Manage Indexes

Use the new index management helpers to inspect indexes created on a video. This replaces the old scene-index-specific management helpers.

In [11]:
indexes = video.list_indexes()

for item in indexes:
    print(item)

Index(index_id=841ed791e6374609, video_id=m-z-019f3ab1-4f0f-7142-b9a8-e9a43b188695, name=scene_quickstart_221d0ecd, status=ready, use_for=['semantic', 'query'], record_count=26)


In [12]:
retrieved_index = video.get_index(index_id=visual_index.index_id)

print("Retrieved index ID:", retrieved_index.index_id)
print("Retrieved index status:", retrieved_index.status)
print("Retrieved index fields:", retrieved_index.fields)

Retrieved index ID: 841ed791e6374609
Retrieved index status: ready
Retrieved index fields: {'filter': ['primary_visual_topic'], 'semantic': ['scene_description', 'primary_visual_topic'], 'text': ['scene_description', 'primary_visual_topic']}


## Scene Index Concepts

A scene index has two stages: first create visual understanding artifacts, then index the fields you want to retrieve. This gives you control over what gets described, embedded, returned, and filtered.

<table>
  <thead>
    <tr>
      <th>Concept</th>
      <th>Where it is configured</th>
      <th>Why it matters</th>
    </tr>
  </thead>
  <tbody>
    <tr>
      <td>Segmentation</td>
      <td><code>segmentation={"type": "time", "seconds": 10}</code></td>
      <td>Controls how the video timeline is split into scene records.</td>
    </tr>
    <tr>
      <td>Frame sampling</td>
      <td><code>sampling={"strategy": "uniform", "frame_count": 2}</code></td>
      <td>Controls which frames represent each segment for VLM description.</td>
    </tr>
    <tr>
      <td>Prompt</td>
      <td>VLM analyzer <code>config["prompt"]</code></td>
      <td>Controls the kind of visual details captured in each record.</td>
    </tr>
    <tr>
      <td>Schema</td>
      <td>VLM analyzer <code>config["schema"]</code></td>
      <td>Produces structured fields such as <code>scene_description</code> and <code>primary_visual_topic</code>.</td>
    </tr>
    <tr>
      <td>Semantic fields</td>
      <td><code>fields["semantic"]</code> in <code>video.index()</code></td>
      <td>Controls which fields power semantic retrieval.</td>
    </tr>
    <tr>
      <td>Text fields</td>
      <td><code>fields["text"]</code> in <code>video.index()</code></td>
      <td>Controls which fields are returned as result metadata.</td>
    </tr>
    <tr>
      <td>Filter fields</td>
      <td><code>fields["filter"]</code> in <code>video.index()</code></td>
      <td>Controls which fields can be used for exact structured filtering.</td>
    </tr>
  </tbody>
</table>

For larger or more specialized workflows, try different prompts, schemas, time windows, and frame counts before creating the final index.

## Further Resources

- [Understanding Artifacts](https://videodb-docs-indexing-search-v2.mintlify.app/pages/understand/indexing-pipelines/understanding-artifacts)
- [Create an Index](https://videodb-docs-indexing-search-v2.mintlify.app/pages/understand/indexing-pipelines/create-an-index)
- [Search and Retrieval](https://videodb-docs-indexing-search-v2.mintlify.app/pages/understand/search-and-retrieval/natural-language-query)
